# 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra kết nối
import os
GDRIVE_PATH = '/content/drive/MyDrive/DLPost'
print("GDrive connected:", os.path.exists(GDRIVE_PATH))
if os.path.exists(GDRIVE_PATH):
    print("Folders:", os.listdir(GDRIVE_PATH))
else:
    print("⚠️  Chưa có thư mục DLPost trên GDrive, sẽ tạo ở bước sau.")

# 2. Clone Git Repo


In [ ]:
import os

REPO_URL = "https://github.com/PNCanh/DLPost.git"
REPO_DIR = "/content/DLPost"

if os.path.exists(REPO_DIR):
    # Nếu đã clone rồi → pull code mới nhất
    %cd {REPO_DIR}
    !git pull origin main
    print("✅ Pulled latest code from GitHub")
else:
    # Clone lần đầu
    !git clone {REPO_URL} {REPO_DIR}
    print("✅ Cloned repo from GitHub")

%cd {REPO_DIR}
!git log --oneline -5  # Xem 5 commit gần nhất

# 3. Install Dependencies


In [ ]:
%cd /content/DLPost
!pip install -r requirements.txt -q
print("✅ Dependencies installed")

# 4. Path Config & Directory Setup


In [ ]:
import sys
import os

# Thêm src vào PYTHONPATH
sys.path.insert(0, '/content/DLPost/src')

# Set environment variable để paths.py detect Colab
# (COLAB_RELEASE_TAG đã có sẵn trên Colab, chỉ cần verify)
print(f"COLAB_RELEASE_TAG: {os.environ.get('COLAB_RELEASE_TAG', 'NOT SET')}")

# Import paths module (tự detect Colab)
from configs import paths
from configs.paths import ensure_directories

# Tạo tất cả thư mục cần thiết
ensure_directories()

# Verify paths
print(f"\n✅ Paths configured (IS_COLAB={paths.IS_COLAB}):")
print(f"   ROOT_DIR:       {paths.ROOT_DIR}")
print(f"   DATASET_DIR:    {paths.DATASET_DIR}")
print(f"   OUTPUT_DIR:     {paths.OUTPUT_DIR}")
print(f"   CHECKPOINTS_DIR:{paths.CHECKPOINTS_DIR}")
print(f"   LOGS_DIR:       {paths.LOGS_DIR}")
print(f"   PREDICTIONS_DIR:{paths.PREDICTIONS_DIR}")
print(f"   FIGURES_DIR:    {paths.FIGURES_DIR}")
print(f"   METRICS_DIR:    {paths.METRICS_DIR}")

# 5. Device Setup


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — using CPU")
    print("    → Vào Runtime > Change runtime type > GPU")

# 6. Select Config & Review

Chọn config để train từ `model_config.py`.

Mỗi config bao gồm: pooling strategy, fusion strategy, loss type, scheduler + warmup.

In [ ]:
# === Option A: Chạy full pipeline (OCR -> Preprocess -> Train) ===
# Chạy tất cả các steps
SELECTED_CONFIG = 'baseline'
!python /content/DLPost/src/main.py --config {SELECTED_CONFIG}

# === Option B: Chạy chỉ training (Bỏ qua OCR và Preprocess) ===
SELECTED_CONFIG = 'baseline'
!python /content/DLPost/src/main.py --config {SELECTED_CONFIG} --skip_ocr --skip_preprocess

In [ ]:
# === Option A: Chạy full pipeline (tất cả steps từ OCR đến evaluate) ===
# Chỉ train các configs trong DEFAULT_CONFIGS (file model_config.py)
!python /content/DLPost/src/main.py

In [ ]:
# === Option B: Chạy chỉ training với config đã chọn ===
import numpy as np
import pandas as pd
from configs import paths
from configs.model_config import TRAINING_CONFIGS
from data.dataset import load_train_dataset, load_val_dataset, load_test_dataset
from data.pytorch_datasets import get_dataloaders
from models.model_factory import ModelFactory
from losses.loss_factory import build_loss
from trainers.trainer_multimodal import MultimodalTrainer

# Load data
train_df = load_train_dataset()
val_df = load_val_dataset()
test_df = load_test_dataset()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

from datetime import datetime

for config_name in SELECTED_CONFIGS:
    config = TRAINING_CONFIGS[config_name].copy()
    
    print(f"\n{'='*60}")
    print(f"=== TRAINING: {config_name} ===")
    print(f"    Pooling:   {config.get('pooling_strategy', 'cls')}")
    print(f"    Fusion:    {config.get('fusion_strategy')}")
    print(f"    Loss:      binary={config['loss']['binary']['type']}, "
          f"multi={config['loss']['multiclass']['type']}")
    print(f"    Gated:     {config['loss'].get('gated', False)}")
    print(f"    Scheduler: {config['training'].get('scheduler', 'none')}")
    print(f"    Warmup:    {config['training'].get('warmup_ratio', 0)}")
    print(f"    LR:        {config['training']['learning_rate']}")
    print(f"    Epochs:    {config['training']['epochs']}")
    print(f"{'='*60}")
    
    # Timestamp và model name
    timestamp = datetime.now().strftime("%M%H%d%m%y")
    model_name = f"{config.get('text_model')}_{config.get('image_model')}_{config.get('fusion_strategy')}"
    config['run_timestamp'] = timestamp
    config['model_name'] = model_name
    
    # Build components
    train_loader, val_loader, test_loader = get_dataloaders(config, train_df, val_df, test_df)
    model = ModelFactory.build_model(config).to(device)
    loss_fn = build_loss(config['loss']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['training']['learning_rate'])
    
    # Train (scheduler + warmup được tự động setup trong BaseTrainer)
    trainer = MultimodalTrainer(
        model=model, optimizer=optimizer, loss_fn=loss_fn,
        train_loader=train_loader, val_loader=val_loader,
        device=device, config=config
    )
    trainer.fit()
    
    # Predict
    predictions = trainer.predict(test_loader)
    print(f"✅ {config_name} completed. {len(predictions)} predictions.")

# 8. Evaluate & Save Results


In [ ]:
from evaluators.evaluator import ClassificationEvaluator
from configs import paths
import json

# Tạo formatted predictions
formatted_preds = []
for i, item in test_df.iterrows():
    if i < len(predictions):
        pred_item = predictions[i]
        if isinstance(pred_item, dict) and "multiclass_probs" in pred_item:
            pred_label = int(np.argmax(pred_item["multiclass_probs"]))
        elif isinstance(pred_item, dict) and "binary_probs" in pred_item:
            pred_label = int(np.argmax(pred_item["binary_probs"]))
        else:
            pred_label = 0
            
        formatted_preds.append({
            "post_id": item.get("post_id", str(i)),
            "true_label": item.get("multi_label", 0),
            "predicted_label": pred_label,
            "explanation_probs": pred_item.get("explanation_probs", []) if isinstance(pred_item, dict) else [],
            "is_scam": pred_label != 0
        })

# Evaluate
evaluator = ClassificationEvaluator(
    run_name=config.get("run_name", "experiment"),
    model_name=model_name,
    timestamp=timestamp
)
mock_metrics = {"train_loss": [], "val_loss": []}
evaluator.evaluate(formatted_preds, mock_metrics)

print(f"\n✅ Results saved:")
print(f"   Checkpoints: {paths.CHECKPOINTS_DIR}")
print(f"   Logs:        {paths.LOGS_DIR}")
print(f"   Predictions: {paths.PREDICTIONS_DIR}")
print(f"   Figures:     {paths.FIGURES_DIR}")
print(f"   Metrics:     {paths.METRICS_DIR}")

# 9. Compare Models (Optional)

So sánh kết quả giữa các configs đã train.

In [ ]:
from evaluators.model_comparator import generate_comparison_report

# Tạo báo cáo so sánh giữa tất cả models đã train
generate_comparison_report()
print("✅ Comparison report generated.")